# 96. 朴素贝叶斯

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 11 / 34 步：扩展监督/无监督模型工具箱**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 支持向量机（SVM）  →  **本章任务：** 朴素贝叶斯  →  **下一步：** K-Means聚类
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

看一个实际问题——给一瓶红酒打上产地标签（class_0、class_1、class_2），手上有几十个化学成分的测量值，该怎么自动做这道多分类选择题？朴素贝叶斯从「每个类别大致服从什么分布」出发，用贝叶斯公式把「先验概率」与「条件概率」组合成「后验概率」，从而给出每个样本属于每个类别的概率，而不只是硬判一个结果。



## 本章目标

学完本章，你将能够：

- **理解**：理解「朴素贝叶斯」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「朴素贝叶斯」的关键输出指标。
- **迁移**：能把「朴素贝叶斯」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 96.1 核心概念

**背景引入**：看一个实际问题——给一瓶红酒打上产地标签（class_0、class_1、class_2），手上有几十个化学成分的测量值，该怎么自动做这道多分类选择题？朴素贝叶斯从「每个类别大致服从什么分布」出发，用贝叶斯公式把「先验概率」与「条件概率」组合成「后验概率」，从而给出每个样本属于每个类别的概率，而不只是硬判一个结果。它训练快、参数少、好解释，正是做基线模型时的顺手工具。

- 朴素贝叶斯假设给定类别后特征条件独立（打个比方：它假定每个特征都“各说各话”、互不商量，像几个同学各报各的分数；这样算得快，但也可能忽略它们其实常一起变。）
- GaussianNB 假设每类中连续特征近似高斯
- 模型速度快且适合做基线
- 分类准确不等于概率已经校准


## 96.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| Wine 多分类基线 | `nb.predict()`、`nb.predict_proba()`、`.fit()` | GaussianNB 无需迭代优化，适合快速建立基准。 | 把条件独立假设当成数据真实机制 |
| 平滑参数比较 | `rows.append()`、`m.score()`、`m.predict_proba()`、`pd.DataFrame()` | var_smoothing 防止方差过小导致数值不稳定。 | 只比较准确率而忽略概率质量 |


## 96.3 示例 1：Wine 多分类基线

GaussianNB 无需迭代优化，适合快速建立基准。


<!-- math-foundation:chapter-96 -->
### 数学推导｜朴素贝叶斯由 Bayes 公式组成

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜从 Bayes 公式开始。** 

$$
P(y\mid x)=\frac{P(x\mid y)P(y)}{P(x)}
$$

比较不同类别时，分母 $P(x)$ 相同，可以忽略。

**第 2 步｜加入条件独立假设。** $P(x\mid y)=\prod_jP(x_j\mid y)$。

**第 3 步｜转到对数空间避免很多小概率连乘下溢。** 

$$
\log score(y)=\log P(y)+\sum_j\log P(x_j\mid y)
$$

最终选择对数得分最大的类别。

**把上面的关系收束为本章计算式：**

$$
P(y\mid x)\propto P(y)\prod_{j=1}^{p}P(x_j\mid y)
$$

**符号解释：** 模型假设给定类别后各特征条件独立。

**代码对应：** 文本频数常用 MultinomialNB，连续近似正态特征常用 GaussianNB。

**使用边界：** 条件独立假设通常不真实，但模型仍可作为快速、可解释的基线。


In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, log_loss

data = load_wine(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=85
)
nb = GaussianNB().fit(X_train, y_train)
pred = nb.predict(X_test)
prob = nb.predict_proba(X_test)
print(
    classification_report(
        y_test, pred, target_names=data.target_names, zero_division=0
    )
)
print("log loss:", round(log_loss(y_test, prob), 3))


**练一练**：把 Wine 数据切分的随机种子从 random_state=85 换成你选的一个整数（比如 random_state=1），重新训练 GaussianNB，看看测试集上的 accuracy 与 log loss 有没有变化。想一想：只改这一处，哪些数字会变、为什么？


In [ ]:
# 请在下方填写代码：只把切分随机种子 random_state 改成一个你选的整数（如 1），
# 重新切分并训练 GaussianNB，计算测试集上的 accuracy 与 log_loss。
#
# 可参考脚手架（去掉注释即可用）：
# X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X, y, stratify=y, random_state=1)
# nb_x = GaussianNB().fit(X_tr2, y_tr2)
# acc_x = nb_x.score(X_te2, y_te2)
# loss_x = log_loss(y_te2, nb_x.predict_proba(X_te2))


In [ ]:
# 完整答案：把切分随机种子改成 1，其余流程不变
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X, y, stratify=y, random_state=1)
nb_x = GaussianNB().fit(X_tr2, y_tr2)
acc_x = nb_x.score(X_te2, y_te2)
loss_x = log_loss(y_te2, nb_x.predict_proba(X_te2))


## 96.4 示例 2：平滑参数比较

var_smoothing 防止方差过小导致数值不稳定。


In [ ]:
import pandas as pd

rows = []
for smoothing in [1e-11, 1e-9, 1e-7, 1e-5]:
    m = GaussianNB(var_smoothing=smoothing).fit(X_train, y_train)
    rows.append(
        [
            smoothing,
            m.score(X_test, y_test),
            log_loss(y_test, m.predict_proba(X_test)),
        ]
    )
display(pd.DataFrame(rows, columns=["var_smoothing", "accuracy", "log_loss"]))


## 96.5 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 96.6 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 96.7 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 96.7.1 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 96.7.2 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 96.8 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 96.8.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 96.9 易错点提醒

- 把条件独立假设当成数据真实机制
- 只比较准确率而忽略概率质量
- 对文本计数使用 GaussianNB 而非 MultinomialNB
- 小测试集上过度调 var_smoothing


## 96.10 练习与作业

1. 设置 priors=[1/3,1/3,1/3]
2. 与数据学习出的类先验比较
3. 报告准确率和 log loss

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 96.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“设置 priors=[1/3,1/3,1/3]”。
2. **独立完成**：不复制示例代码，完成“与数据学习出的类先验比较”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“报告准确率和 log loss”，用一两句话说明你修改了什么。

### 96.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 96.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
uniform_nb = GaussianNB(priors=[1 / 3, 1 / 3, 1 / 3]).fit(X_train, y_train)
uniform_prob = uniform_nb.predict_proba(X_test)
practice_accuracy = uniform_nb.score(X_test, y_test)
practice_loss = log_loss(y_test, uniform_prob)
print(round(practice_accuracy, 3), round(practice_loss, 3))


## 96.12 小结

使用 Gaussian Naive Bayes 建立快速概率分类基线，理解条件独立假设和概率平滑。


### 96.12.1 你已经掌握

- 训练 GaussianNB
- 理解类先验与条件似然
- 获取预测概率
- 用校准曲线检查概率质量


### 96.12.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 96.12.3 需要注意

- 把条件独立假设当成数据真实机制
- 只比较准确率而忽略概率质量
- 对文本计数使用 GaussianNB 而非 MultinomialNB
- 小测试集上过度调 var_smoothing


### 96.12.4 完成检查

- [ ] 能够训练 GaussianNB
- [ ] 能够理解类先验与条件似然
- [ ] 能够获取预测概率
- [ ] 能够用校准曲线检查概率质量


### 96.12.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
